# Suite CON — Constraints

A constraint is a named Basic rule about the instances of one schema, kept beside the schemas; a set gathers
constraints. Both are data: bound classes of their meta-schemas, stored and read back with their rules.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Framework import Terms
from mbse.Patterns import Constraints as C, register
from mbse.Schemas.Framework import JSON, Proxies, Validators as SV
from support import raises

this = E.variable("this")
adult = C.Constraint("Contact", "adult", this.age.ge(18), "of age")
named = C.Constraint("Contact", "named", this.has("name"))
numbered = C.Constraint("Phone", "numbered", this.has("number"))

## CON-01 · A constraint is a named rule about one schema's instances

In [ ]:
assert (adult.schema, adult.name, adult.description) == ("Contact", "adult", "of age")
assert isinstance(adult.rule, E.OfOperation.Data) and named.description is None  # a writer is resolved to its data
assert C.Constraint("Contact", "same", adult.rule).rule is adult.rule  # data is taken as it is
assert (adult.schema_name(), adult.owner(), adult.identity() == adult.identity()) == ("Patterns.Constraint", None, True)
assert adult.validate() == [] and C.Constraint("Contact", "late", None).rule is None

## CON-02 · Constraints are checked statically

In [ ]:
bad = C.Constraint("Contact", "x", E.operation("pow", this.age, E.variable("n")))
assert bad.validate() == ["constraint 'x' of 'Contact': 'pow' is not a core operation",
                          "constraint 'x' of 'Contact': argument 1: variable 'n' is not bound"]
assert C.Constraint("", "", None).validate() == ["constraint '' of '': a constraint needs a schema",
                                                 "constraint '' of '': a constraint needs a name",
                                                 "constraint '' of '': a constraint needs a rule"]
twice = C.Set([adult, named, C.Constraint("Contact", "adult", this.age.ge(21))])
assert twice.validate() == ["constraint 'adult' of 'Contact': defined twice"]
with raises(ValueError, match="'pow' is not a core operation; constraint 'x' of 'Contact': argument 1"):
    C.check([bad])
constraints = C.check([adult, named, numbered])  # a set, from any constraints
assert isinstance(constraints, C.Set) and C.check(constraints) is constraints

## CON-03 · A set orders constraints and gives each schema's

In [ ]:
constraints = C.Set([adult, numbered, named])
assert constraints.constraints == (adult, numbered, named)  # a tuple, in order
assert constraints.of("Contact") == (adult, named) and constraints.of("Phone") == (numbered,) and constraints.of("X") == ()
assert C.Set().constraints == () and constraints.schema_name() == "Patterns.Set" and constraints.owner() is None
assert constraints.identity() == constraints.identity() != C.Set().identity()

## CON-04 · A set is data, read back with its rules

In [ ]:
shared = this.age.ge(18).data
constraints = C.Set([C.Constraint("Contact", "adult", shared, "of age"),
                     C.Constraint("Contact", "phoned", E.operation("implies", shared, this.has("phones")))])
text = JSON.ToJSON(C.Builders).Reachable(C.Set.Schema, constraints)
assert text.count('"kind": "operation"') == 4  # get, ge, implies and has: the shared rule's two are written once
copy = JSON.FromJSON(C.Builders).Reachable(C.Set.Schema, text)
assert [(c.schema, c.name, c.description) for c in copy.constraints] == [("Contact", "adult", "of age"),
                                                                         ("Contact", "phoned", None)]
assert Terms.same(copy.constraints[1].rule, constraints.constraints[1].rule)
assert copy.constraints[1].rule.arguments[0] is copy.constraints[0].rule  # still shared
assert SV.Validate(C.Builders).Reachable(C.Set.Schema, constraints) == []  # valid data, by mbse-schemas' rules
built = C.Builders.builder("Patterns.Constraint", adult)
built.property("name", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("grown"))))
assert built.update() is adult and adult.name == "grown" and adult.rule is not None
adult.name = "adult"

## CON-05 · Decoding refuses what a constraint or a set cannot hold

In [ ]:
def rule_entry(target, index=0):
    def fill(e):
        if target is not None:
            e.link("argument", lambda l: l.set(target))
        e.property("index", lambda p: p.value(lambda v: v.as_native(lambda n: n.set(index))))
    return lambda a: a.add(fill)


def member_entry(target, index=None):
    def fill(e):
        if target is not None:
            e.link("constraint", lambda l: l.set(target))
        if index is not None:
            e.property("index", lambda p: p.value(lambda v: v.as_native(lambda n: n.set(index))))
    return lambda a: a.add(fill)


with raises(ValueError, match="a constraint has one rule, got 2"):
    C.Builders.builder("Patterns.Constraint").adjacency("rule", rule_entry(this.data)).adjacency(
        "rule", rule_entry(this.data, 1)).create()
with raises(TypeError, match="a rule must be a Basic expression"):
    C.Builders.builder("Patterns.Constraint").adjacency("rule", rule_entry(named)).create()
with raises(ValueError, match="link 'argument' is not set"):
    C.Builders.builder("Patterns.Constraint").adjacency("rule", rule_entry(None)).create()
with raises(TypeError, match="a member must be a constraint"):
    C.Builders.builder("Patterns.Set").adjacency("constraints", member_entry(this.data, 0)).create()
with raises(ValueError, match="link 'constraint' is not set"):
    C.Builders.builder("Patterns.Set").adjacency("constraints", member_entry(None, 0)).create()
unordered = C.Builders.builder("Patterns.Set").adjacency("constraints", member_entry(named)).adjacency(
    "constraints", member_entry(adult, 0)).adjacency("constraints", member_entry(numbered)).create()
assert unordered.constraints == (adult, named, numbered)  # entries without an index come last, in their order
blank = C.Builders.builder("Patterns.Constraint").create()  # what a snapshot leaves out is empty, and validate() says so
assert (blank.schema, blank.name, blank.rule, blank.description) == ("", "", None, None)

## CON-06 · Another store holds constraints as proxies

In [ ]:
store = register(Proxies.OfStore())
assert {"Patterns.Constraint", "Patterns.Set", "Patterns.Members", "Expressions.OfOperation"} <= set(store.names())
assert register(store) is store  # what it holds already is skipped
proxy = JSON.FromJSON(store).Reachable(C.Set.Schema, JSON.ToJSON(C.Builders).Reachable(C.Set.Schema, C.Set([adult])))
assert proxy.schema_name() == "Patterns.Set" and Proxies.store_of(proxy) is store